# Notebook 01 — Exploratory Data Analysis & Visualization
## Ames Housing Dataset

This notebook explores the raw Ames Housing data to understand its structure,
target distribution, missing value patterns, and key predictors — before any
cleaning or modelling begins.

**Workflow position:** This is the first step. Nothing is modified; we only look.

**Output:** Documented understanding of the dataset that drives all subsequent
cleaning and modelling decisions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

📝 **What's happening here** — Standard imports. `seaborn` for statistical plots, `matplotlib` for custom figures. `train_test_split` is imported here so we can verify split quality (Section 3) using the same split we will use in Notebook 02.

## 1) Load Dataset

Ames Housing contains 2,930 residential property sales in Ames, Iowa.
80 features cover size, quality, age, neighborhood, garage, basement, and more.
Target: `SalePrice` (continuous regression).

In [ ]:
# Load from local CSV — column names use spaces; we normalise them to CamelCase
df = pd.read_csv("../data/raw/AmesHousing.csv")
df.columns = df.columns.str.strip().str.replace(' ', '').str.replace('/', '')

# Drop row-ID columns not useful for modelling
df = df.drop(columns=['Order', 'PID'], errors='ignore')

# Confirm target is numeric
df['SalePrice'] = pd.to_numeric(df['SalePrice'], errors='coerce')

print(f'Shape: {df.shape}')
df.head()

📝 **What's happening here** — `AmesHousing.csv` header uses spaces (e.g. `'Lot Frontage'`). `.str.replace(' ', '')` converts them to CamelCase (`'LotFrontage'`), matching the column names used everywhere else in the project. `Order` and `PID` are row identifiers, not predictors — drop them early so they can't accidentally enter any model.

In [ ]:
# Column type overview
print('Column type breakdown:')
print(df.dtypes.value_counts())
print(f'\nNumeric columns : {df.select_dtypes(include="number").shape[1]}')
print(f'Categorical cols: {df.select_dtypes(include="object").shape[1]}')

📝 **What's happening here** — The dtype split tells us what kind of work lies ahead. Numeric columns need scaling; categorical columns need encoding. Knowing the count of each type before touching the data helps plan the preprocessing pipeline.

In [ ]:
# Summary statistics for numeric columns
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

📝 **What's happening here** — A heat-mapped transposed `describe()` lets us scan ranges at a glance. Large spread in `mean` and `std` across features (e.g. `LotArea` vs `OverallQual`) signals that features live on very different scales — this motivates StandardScaler in Notebook 02.

In [ ]:
# Categorical column cardinality + sample values
cat_cols = df.select_dtypes(include='object').columns
cat_cardinality = df[cat_cols].nunique().sort_values(ascending=False)

print('Categorical column cardinality (unique values):')
print(cat_cardinality.to_string())

# Sample values for key columns
print()
sample_cols = ['OverallQual', 'BsmtQual', 'KitchenQual', 'GarageType', 'Neighborhood']
for col in sample_cols:
    if col in df.columns:
        print(f'{col}: {df[col].value_counts().head(5).to_dict()}')

📝 **What's happening here** — Cardinality tells us how many OHE columns each categorical will generate. `Neighborhood` has 25 unique values → 25 binary columns. Columns with cardinality 2–5 are candidates for ordinal encoding. The sample value counts reveal dominant categories (e.g. `KitchenQual` is mostly `'TA'`) which will guide the imputation mode choice in Notebook 02.

## 2) Target Variable Analysis

Before splitting, understand the shape of what we are predicting.
A right-skewed target can affect model performance and metric interpretation.

In [ ]:
print('SalePrice summary:')
print(df['SalePrice'].describe().round(0))
print(f'\nSkewness: {df["SalePrice"].skew():.3f}')
print(f'Kurtosis: {df["SalePrice"].kurt():.3f}')

plt.figure(figsize=(8, 4))
sns.histplot(df['SalePrice'], bins=50, kde=True)
plt.axvline(df['SalePrice'].mean(),   color='red',   linestyle='--',
            label=f'Mean  ${df["SalePrice"].mean():,.0f}')
plt.axvline(df['SalePrice'].median(), color='green', linestyle='--',
            label=f'Median ${df["SalePrice"].median():,.0f}')
plt.title('SalePrice Distribution')
plt.xlabel('SalePrice ($)')
plt.legend()
plt.tight_layout()
plt.show()

📝 **What's happening here** — Skewness > 1 means a pronounced right tail — a few expensive houses pull the mean above the median. The histogram makes this visible: most homes cluster under $250k, but a long tail extends to $750k+. Models trained on a skewed target tend to make larger dollar errors on expensive houses. A log transform (explored next) can fix this.

In [ ]:
# Log-transform comparison: raw vs log1p(SalePrice)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['SalePrice'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('SalePrice (raw)  — right-skewed')
axes[0].set_xlabel('SalePrice ($)')

sns.histplot(np.log1p(df['SalePrice']), bins=50, kde=True, ax=axes[1], color='orange')
axes[1].set_title('log1p(SalePrice)  — more symmetric')
axes[1].set_xlabel('log(SalePrice + 1)')

plt.tight_layout()
plt.show()

print('Note: Notebook 03 trains on log1p(SalePrice) for better-behaved residuals.')

📝 **What's happening here** — `log1p(x) = log(x + 1)` compresses the right tail, producing a roughly bell-shaped distribution. Models trained on the log scale minimise percentage error rather than absolute dollar error — a better fit for housing data where a $50k error on a $100k house is far worse than on a $500k house. Predictions are exponentiated back with `np.expm1()` at evaluation time.

In [ ]:
# SalePrice quartile breakdown
quartiles = pd.qcut(df['SalePrice'], q=4,
                    labels=['Q1 (cheap)', 'Q2', 'Q3', 'Q4 (expensive)'])
print('SalePrice quartile ranges:')
print(df.groupby(quartiles, observed=True)['SalePrice'].agg(['min', 'max', 'count']))

📝 **What's happening here** — Cutting into equal-count quartiles tells us the price ranges that separate the dataset into fourths. Q4 spans the widest dollar range (evidence of the right skew). This segmentation is useful when evaluating whether the model performs equally well across price tiers.

## 3) Train / Validation / Test Split

We verify split quality here as an EDA step. The **same split** is reproduced
in Notebook 02 (with `random_state=42`) for cleaning and preprocessing.

| Split | Size | Role |
|-------|------|------|
| Train | 60% | Model + preprocessor fitting |
| Validation | 20% | Model comparison and tuning |
| Test | 20% | Final honest evaluation (touch once) |

In [ ]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

# Step 1: 60% train, 40% temporary holdout
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42
)
# Step 2: split the 40% holdout evenly → 20% val, 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

total = len(df)
print(f'Train:      {len(X_train):5d} rows  ({len(X_train)/total*100:.0f}%)')
print(f'Validation: {len(X_val):5d} rows  ({len(X_val)/total*100:.0f}%)')
print(f'Test:       {len(X_test):5d} rows  ({len(X_test)/total*100:.0f}%)')
print(f'Total:      {total:5d} rows')

📝 **What's happening here** — `train_test_split` only creates two subsets, so we call it twice: first to carve off 40%, then to halve that into val and test. `random_state=42` ensures every run produces the identical split — critical for reproducibility. The three-way split (rather than 80/20) keeps the test set genuinely unseen throughout all modelling decisions.

In [ ]:
# Compare SalePrice statistics across splits
split_comparison = pd.DataFrame({
    'Train':      y_train.describe(),
    'Validation': y_val.describe(),
    'Test':       y_test.describe()
}).round(0)

print('SalePrice distribution per split:')
print(split_comparison)

# Overlay SalePrice distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name, series, color in [('Train', y_train, 'steelblue'),
                              ('Val',   y_val,   'orange'),
                              ('Test',  y_test,  'green')]:
    axes[0].hist(series, bins=40, alpha=0.45, label=name, color=color, density=True)
axes[0].set_title('SalePrice Distribution Overlay')
axes[0].set_xlabel('SalePrice ($)')
axes[0].legend()

split_df = pd.DataFrame({
    'SalePrice': pd.concat([y_train, y_val, y_test]),
    'Split': (['Train'] * len(y_train) + ['Val'] * len(y_val) + ['Test'] * len(y_test))
})
sns.boxplot(data=split_df, x='Split', y='SalePrice', ax=axes[1],
            order=['Train', 'Val', 'Test'])
axes[1].set_title('SalePrice Box Plot per Split')

plt.tight_layout()
plt.show()
print('The three splits look similar — the random split is balanced.')

📝 **What's happening here** — A good split has similar target distributions across all three sets. Check that means and medians are within ~10% of each other; the overlaid histogram shapes should be nearly identical. If one split had a drastically different distribution, evaluation metrics would be misleading regardless of model quality.

## 4) Missing Value Audit

Always audit missing values on the **training set only**. The validation and
test sets represent future data — never use them to make preprocessing decisions.

In [ ]:
# Null summary on training set only
null_counts = X_train.isnull().sum()
null_pct    = (null_counts / len(X_train) * 100).round(1)

null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_pct':   null_pct,
    'dtype':      X_train.dtypes
}).query('null_count > 0').sort_values('null_pct', ascending=False)

print(f'Columns with missing values: {len(null_summary)}')
null_summary

📝 **What's happening here** — We build the null audit from `X_train` only. The `null_pct` column reveals which features are nearly always missing (structural absence — no pool, no alley) versus occasionally missing (recording gaps). The `dtype` column drives the imputation strategy: categorical columns get a fill *label* (`'None'` or mode), numeric columns get a fill *number* (0 or median).

In [ ]:
# Missing value heatmap (top 15 null columns, train set)
top_null_cols = null_summary.head(15).index.tolist()

plt.figure(figsize=(13, 5))
sns.heatmap(X_train[top_null_cols].isnull(), cbar=False, yticklabels=False,
            cmap='Reds')
plt.title('Missing Value Heatmap — Top 15 Null Columns (train set)')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

📝 **What's happening here** — The heatmap reveals *where* nulls occur, not just *how many*. Look for vertical bands (an entire column missing — e.g. `PoolQC`) versus scattered horizontal patterns (the same rows missing across multiple columns). Coordinated row-level patterns indicate that those columns describe the same absent feature (no garage, no basement).

In [ ]:
# Bar chart of null percentages
plt.figure(figsize=(12, 5))
null_summary['null_pct'].sort_values(ascending=True).plot(
    kind='barh', color='salmon', edgecolor='white'
)
plt.axvline(50, color='red', linestyle='--', alpha=0.6, label='50% threshold')
plt.title('Missing Value Percentage by Column (train set)')
plt.xlabel('% Missing')
plt.legend()
plt.tight_layout()
plt.show()

# Null count by dtype
print('Null columns by dtype:')
print(null_summary.groupby('dtype')['null_count'].count())

📝 **What's happening here** — The bar chart gives an at-a-glance ranking. Columns above 50% missing are *structural absences* — the null is the information (e.g. `PoolQC` null → no pool). The dtype split at the bottom separates the imputation problem: numeric nulls get a numeric fill, categorical nulls get a label.

## 5) Null Co-occurrence Analysis

Some columns are null on the **same rows** because they all describe the same
absent feature. Identifying these groups confirms structural interpretation.

In [ ]:
# Garage-related columns — are they null on the same rows?
garage_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
               'GarageYrBlt', 'GarageArea', 'GarageCars']
garage_null = X_train[garage_cols].isnull()

overlap = garage_null.T.dot(garage_null)
print('Garage null overlap (rows where each pair is BOTH null):')
print(overlap)
print(f'\nRows with any garage null:  {garage_null.any(axis=1).sum()}')
print(f'Rows with ALL garage cols null: {garage_null.all(axis=1).sum()}')

# Basement-related columns
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath']
bsmt_null = X_train[bsmt_cols].isnull()
print('\nBasement co-occurrence:')
print(f'Rows where ALL bsmt quality cols null: '
      f'{bsmt_null[["BsmtQual","BsmtCond","BsmtExposure"]].all(axis=1).sum()}')
print('Conclusion: categorical bsmt cols null = no basement → fill with None / 0.')

📝 **What's happening here** — The overlap matrix confirms that garage-null rows are null on *every* garage column simultaneously. This is a coordinated, domain-consistent absence — not a random data-recording failure. The correct fill for `GarageYrBlt` is `0` (no year built because no garage exists), not the median year — filling with the median would internally contradict `GarageArea = 0` on the same row.

In [ ]:
# Structural null co-occurrence correlation heatmap
structural_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
present = [c for c in structural_cols if c in X_train.columns]
null_indicator = X_train[present].isnull().astype(int)

plt.figure(figsize=(10, 6))
sns.heatmap(null_indicator.corr(), annot=False, cmap='coolwarm', center=0,
            linewidths=0.5)
plt.title('Null Co-occurrence Correlation — Structural Columns')
plt.tight_layout()
plt.show()
print('High correlation = columns are null on the same rows = same absent feature.')

📝 **What's happening here** — The correlation heatmap of the null-indicator matrix shows which column pairs tend to be missing together. A block of high positive correlation among garage columns (and another for basement columns) confirms the structural interpretation. This visualisation is the evidence base for the four-bucket imputation strategy in Notebook 02.

## 6) Feature Type Classification

Classifying features into numeric, ordinal, and nominal groups is required
before building the sklearn `ColumnTransformer` in Notebook 03.

In [ ]:
# Ordinal columns: categories have a meaningful rank (quality/condition scales)
ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual', 'FireplaceQu',
    'GarageQual', 'GarageCond', 'PoolQC'
]

# All other categorical columns → one-hot encoding
nominal_cols = [
    c for c in X_train.select_dtypes(include='object').columns
    if c not in ordinal_cols
]

# Numeric columns (includes ordinal-as-integer like OverallQual)
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f'Numeric cols : {len(numeric_cols):3d}  (sample: {numeric_cols[:4]})')
print(f'Ordinal cols : {len(ordinal_cols):3d}  (sample: {ordinal_cols[:4]})')
print(f'Nominal cols : {len(nominal_cols):3d}  (sample: {nominal_cols[:4]})')

📝 **What's happening here** — The three-way classification drives the entire preprocessing pipeline. **Ordinal** columns use `OrdinalEncoder` with an explicit quality ordering (`Po < Fa < TA < Gd < Ex`) so the model receives a monotonic numeric signal. **Nominal** columns use `OneHotEncoder` since their categories have no inherent rank. **Numeric** columns pass through imputation and scaling only. Any column not assigned to a group will be silently dropped by `ColumnTransformer` — a common silent bug to guard against.

## 7) Correlation Analysis

Pearson correlation measures the linear relationship between numeric features
and `SalePrice`. High-correlation features are strong candidates for the
final 10-feature selection in Notebook 03.

In [ ]:
corr_cols = numeric_cols + ['SalePrice']
corr_matrix = df[[c for c in corr_cols if c in df.columns]].corr().round(2)

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
            linewidths=0.3, square=False)
plt.title('Pearson Correlation — Numeric Features + SalePrice')
plt.tight_layout()
plt.show()

📝 **What's happening here** — The full correlation heatmap shows inter-feature relationships as well as target correlations. Strong off-diagonal blocks (e.g. garage-related features correlating with each other) reveal multicollinearity — two features carrying the same information. Notebook 03 addresses this with VIF checks before finalising the feature set.

In [ ]:
corr_with_target = corr_matrix['SalePrice'].drop('SalePrice')
top15 = corr_with_target.abs().sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 6))
top15.plot(kind='barh', color='steelblue')
plt.axvline(0.5, color='red', linestyle='--', alpha=0.6, label='r = 0.5 threshold')
plt.gca().invert_yaxis()
plt.xlabel('|Pearson r with SalePrice|')
plt.title('Top 15 Numeric Features by Correlation with SalePrice')
plt.legend()
plt.tight_layout()
plt.show()

print('Top 5 predictors by Pearson r:')
print(top15.head())

📝 **What's happening here** — Features above r = 0.5 have a strong linear relationship with price. `OverallQual` typically leads (~0.79), followed by `GrLivArea`, `GarageCars`, and `TotalBsmtSF`. These will also rank highly in Gradient Boosting feature importances — cross-validating a linear metric against a nonlinear model on the same features confirms the signal is robust.

## 8) Categorical & Ordinal Feature Analysis

Box and violin plots show how price varies across categorical categories,
revealing which nominal and ordinal features carry the most predictive signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Neighborhood median price (sorted)
nbhd_order = (df.groupby('Neighborhood')['SalePrice']
                .median()
                .sort_values()
                .index)
sns.boxplot(data=df, x='SalePrice', y='Neighborhood',
            order=nbhd_order, ax=axes[0], orient='h')
axes[0].set_title('SalePrice by Neighborhood (sorted by median)')
axes[0].set_xlabel('SalePrice ($)')

# OverallQual vs SalePrice
sns.boxplot(data=df, x='OverallQual', y='SalePrice', ax=axes[1],
            palette='viridis')
axes[1].set_title('SalePrice by Overall Quality')
axes[1].set_xlabel('Overall Quality (1–10)')

plt.tight_layout()
plt.show()

📝 **What's happening here** — `Neighborhood` is the highest-cardinality nominal feature (25 unique values). Sorting by median price reveals a clear price gradient — top neighbourhoods like `NridgHt` command nearly 2× the median price of bottom ones like `MeadowV`. This strong signal means `Neighborhood` must stay in the feature set despite producing 25 OHE columns. `OverallQual` shows a nearly monotonic relationship: each quality point adds roughly $20k in median price.

In [ ]:
# Violin plots for ordinal quality features
quality_order = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
ordinal_plot_cols = ['ExterQual', 'KitchenQual', 'BsmtQual']
present_ord = [c for c in ordinal_plot_cols if c in df.columns]

fig, axes = plt.subplots(1, len(present_ord), figsize=(14, 5))

for ax, col in zip(axes, present_ord):
    plot_df = df[df[col].isin(quality_order)].copy()
    order_present = [v for v in quality_order if v in plot_df[col].unique()]
    sns.violinplot(data=plot_df, x=col, y='SalePrice',
                   order=order_present, ax=ax,
                   palette='plasma', inner='box')
    ax.set_title(col)
    ax.set_xlabel('Quality Level')
    ax.set_ylabel('SalePrice ($)')

plt.suptitle('SalePrice by Ordinal Quality Features', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

📝 **What's happening here** — Violin plots show both the distribution shape and median across quality levels. The consistent leftward-to-rightward price increase (`Po → Fa → TA → Gd → Ex`) justifies ordinal encoding with an explicit order — the model will receive a single numeric axis (0–5) that correctly encodes the rank. If we used OHE instead, this monotonic relationship would be lost.

## 9) Outlier Detection

Extreme observations in `GrLivArea` are well-known in the Ames Housing dataset:
two very large houses sold for unusually low prices, likely due to partial sales.
Notebook 02 removes them from the training set only.

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df['GrLivArea'], df['SalePrice'],
            alpha=0.35, s=12, color='steelblue', label='Normal')

# Flag the two known outliers: large area, low price
outlier_mask = (df['GrLivArea'] > 4000) & (df['SalePrice'] < 200_000)
plt.scatter(df.loc[outlier_mask, 'GrLivArea'],
            df.loc[outlier_mask, 'SalePrice'],
            color='red', s=60, zorder=5, label='Outliers to remove')

plt.xlabel('Above-Grade Living Area (sqft)')
plt.ylabel('SalePrice ($)')
plt.title('GrLivArea vs SalePrice — Outlier Detection')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Outlier rows: {outlier_mask.sum()} — will be removed from training only.')

📝 **What's happening here** — The two red points are partial-sale properties that don't represent typical market value. Including them distorts the model's coefficient for living area — the most important predictor. They are removed from `X_train` only (never from val or test), because val/test simulate future real-world data that we do not get to filter in production.

## 10) Key Findings

| Finding | Implication |
|---------|-------------|
| 2,930 rows × 80 features | Large enough for 60/20/20 split |
| SalePrice skewness > 1 | Log-transform the target in Notebook 03 |
| 19 columns with nulls | Require domain-driven imputation (Notebook 02) |
| Garage + basement nulls co-occur | Fill with `'None'` / `0` (structural absence) |
| `LotFrontage` 15% null | Fill with median (measurement gap) |
| Ordinal quality features monotonic | Use `OrdinalEncoder`, not OHE |
| `Neighborhood` 25 unique values | Keep — strong price gradient |
| Two GrLivArea outliers | Remove from train only |
| Top numeric predictors | `OverallQual`, `GrLivArea`, `GarageCars`, `TotalBsmtSF`, `YearBuilt` |

**Next step:** Notebook 02 — Data Cleaning & Preprocessing